In [ ]:
# Instant: UTC-based unambiguous timestamp construction + arithmetic.
from whenever import Instant

t0 = Instant.from_utc(2026, 6, 8, hour=14, minute=30, second=0)
print(f"t0 = {t0}")
print(f"t0.timestamp() = {t0.timestamp()}")

now = Instant.now()
print(f"now type = {type(now).__name__}")

# Round-trip via ISO 8601 string (format_iso/parse_iso in whenever >= 0.9)
iso = t0.format_iso()
t1 = Instant.parse_iso(iso)
assert t0 == t1, f"round-trip failed: {t0!r} != {t1!r}"
print(f"ISO round-trip OK: {iso}")

In [ ]:
# ZonedDateTime: IANA timezone-aware datetime.
from whenever import ZonedDateTime

dt = ZonedDateTime(2026, 12, 25, hour=9, minute=0, tz="America/New_York")
print(f"dt = {dt}")
print(f"tz = {dt.tz}")
print(f"offset = {dt.offset}")

# Convert to a different zone
dt_tokyo = dt.to_tz("Asia/Tokyo")
print(f"dt_tokyo = {dt_tokyo}")

# Same Instant, different wall clocks
assert dt.to_instant() == dt_tokyo.to_instant()
print("ZonedDateTime conversions preserve Instant")

In [ ]:
# PlainDateTime / Date / Time: civil (timezone-naive) types.
from whenever import PlainDateTime, Date, Time

d = Date(2026, 6, 8)
t = Time(14, 30, 0)
pdt = d.at(t)  # Date.at(Time) -> PlainDateTime
assert isinstance(pdt, PlainDateTime)
print(f"date = {d}")
print(f"time = {t}")
print(f"plain = {pdt}")

# Parse ISO
d2 = Date.parse_iso("2026-06-08")
assert d2 == d

# day_of_week is an enum
print(f"day_of_week = {d.day_of_week()}")
print("PlainDateTime / Date / Time OK")

In [ ]:
# Durations: TimeDelta and DateDelta arithmetic.
from whenever import Instant, hours, minutes

t0 = Instant.from_utc(2026, 1, 1, hour=0, minute=0, second=0)
t1 = t0 + hours(48) + minutes(30)
print(f"t0 = {t0}")
print(f"t1 = {t1}")

delta = t1 - t0
print(f"delta = {delta}")
print(f"delta total seconds = {delta.total('seconds')}")

expected_seconds = 48 * 3600 + 30 * 60
assert delta.total('seconds') == expected_seconds, f"{delta.total('seconds')} != {expected_seconds}"
print(f"TimeDelta arithmetic OK ({expected_seconds}s)")

In [ ]:
# Interop with stdlib datetime + RFC 2822 round-trip.
from datetime import datetime, timezone
from whenever import Instant, ZonedDateTime

# stdlib -> whenever (Instant() accepts an aware stdlib datetime directly)
py_dt = datetime(2026, 6, 8, 14, 30, 0, tzinfo=timezone.utc)
inst = Instant(py_dt)
print(f"Instant(py_dt) = {inst}")

# whenever -> stdlib
py_dt2 = inst.to_stdlib()
assert py_dt2 == py_dt, f"{py_dt2!r} != {py_dt!r}"

# RFC 2822 round-trip (rfc3339 helpers were folded into format_iso in whenever >= 0.9)
rfc = inst.format_rfc2822()
inst2 = Instant.parse_rfc2822(rfc)
assert inst == inst2
print(f"RFC2822 = {rfc}")

# ZonedDateTime stdlib round-trip
zdt = ZonedDateTime(2026, 6, 8, hour=14, minute=30, tz="UTC")
assert zdt.to_instant() == inst
print("stdlib interop + RFC 2822 OK")